In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Pick a window and an overlap. The blue curves are the windows of successive
frames, each shifted by the hop length $N_H$, and the red curve is their sum.
Between the dotted lines (away from the edges), the readout checks whether
that sum is a constant $c$, the COLA condition. Try the Hann window at every
overlap, then the rectangular window at $-25\%$.

In [ ]:
# hide
# autorun
WINDOW0, OVERLAP0 = "Hann", 0.5     # the chapter's example

N_F = 64                            # frame length, in samples
SPAN = 10 * N_F                     # frames start within the first SPAN samples
OVERLAPS = [("-25%", -0.25), ("0%", 0.0), ("12.5%", 0.125), ("25%", 0.25),
            ("37.5%", 0.375), ("50%", 0.5), ("62.5%", 0.625), ("75%", 0.75),
            ("87.5%", 0.875)]


def window(shape, N_F=N_F):
    n = np.arange(N_F)
    if shape == "Hann":
        return 0.5 * (1 - np.cos(2 * np.pi * n / N_F))
    if shape == "Triangular":
        return 1 - np.abs(2 * n / N_F - 1)
    return np.ones(N_F)


def overlap_add(shape, overlap, window=window, N_F=N_F, SPAN=SPAN):
    """The shifted windows (one trace, gaps between them) and their sum."""
    N_H = round(N_F * (1 - overlap))
    w = window(shape)
    starts = np.arange(0, SPAN + 1, N_H)
    total = np.zeros(SPAN + N_F + N_H)
    xs, ys = [], []
    for s in starts:
        total[s:s + N_F] += w
        xs += [np.arange(s, s + N_F), [np.nan]]
        ys += [w, [np.nan]]
    # the steady state: every window that can cover these samples exists
    steady = total[N_F:SPAN + 1]
    return N_H, np.concatenate(xs), np.concatenate(ys), total[:SPAN + N_F], steady


def figure():
    N_H, xs, ys, total, steady = overlap_add(WINDOW0, OVERLAP0)
    fig = go.Figure()
    fig.add_scatter(x=[N_F, N_F, None, SPAN, SPAN], y=[0, 9, None, 0, 9],
                    mode="lines", line=dict(color=STEEL, width=1, dash="dot"))
    fig.add_scatter(x=xs, y=ys, mode="lines", opacity=0.55,
                    line=dict(color=BLUE, width=1.4))
    fig.add_scatter(x=np.arange(len(total)), y=total, mode="lines",
                    line=dict(color=RED, width=3))
    fig.update_xaxes(range=[0, SPAN + N_F], title_text="Sample index n",
                     fixedrange=True)
    fig.update_yaxes(range=[0, max(1.15, 1.12 * total.max())],
                     title_text="Window value", fixedrange=True)
    return fig


def controls(fig):
    shape = widgets.Dropdown(description="Window",
                             options=["Rectangular", "Hann", "Triangular"],
                             value=WINDOW0)
    overlap = widgets.SelectionSlider(description="Overlap", options=OVERLAPS,
                                      value=OVERLAP0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(shape, overlap, overlap_add=overlap_add, N_F=N_F, readout=readout):
        N_H, xs, ys, total, steady = overlap_add(shape, overlap)
        with fig.batch_update():
            fig.data[1].x, fig.data[1].y = xs, ys
            fig.data[2].y = total
            fig.layout.yaxis.range = [0, max(1.15, 1.12 * total.max())]
        lo, hi = steady.min(), steady.max()
        head = f"<i>N</i><sub>F</sub> = {N_F}, <i>N</i><sub>H</sub> = {N_H} samples"
        if lo < 1e-9 and N_H > N_F:
            text = ("gaps: some samples are covered by no window at all, "
                    "so overlap-add loses them")
        elif lo < 1e-9:
            text = ("not COLA: where each window tapers to zero, no other "
                    "window covers the sample, so overlap-add loses it")
        elif hi - lo < 1e-9 * hi:
            text = (f"COLA: between the dotted lines the windows sum to "
                    f"<i>c</i> = {hi:.2f}, so dividing by <i>c</i> reconstructs "
                    f"the signal exactly")
        else:
            mid = (hi + lo) / 2
            text = (f"not COLA: the sum ripples between {lo:.2f} and {hi:.2f} "
                    f"(&plusmn;{100 * (hi - lo) / 2 / mid:.0f}%), so overlap-add "
                    f"would add a wobble in amplitude")
        readout.value = (f"<span style='font-size:0.9em'>{head} &nbsp;·&nbsp; "
                         f"{text}</span>")

    widgets.interactive_output(update, {"shape": shape, "overlap": overlap})
    return widgets.VBox([shape, overlap, readout])

icm_plotly.show(figure, controls)